In [ ]:
!pip install marker-pdf

In [ ]:
%%writefile extract_nios.py
import os
import json
import shutil
import re
import time
from pathlib import Path

# Enforce Kaggle GPU constraints
os.environ["INFERENCE_RAM"] = "15" 
os.environ["VRAM_PER_TASK"] = "3"  
os.environ["TORCH_DEVICE"] = "cuda"

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
from marker.output import text_from_rendered

INPUT_DIR = "/kaggle/input/datasets/dipankaj/nios-chapter-pdfs"
OUTPUT_DIR = "/kaggle/working/nios-extracted"
LEDGER_FILE = "/kaggle/working/progress_ledger.json"

# 10 hours in seconds. Safely within Kaggle's 12-hour limit.
MAX_RUNTIME_SECONDS = 10 * 60 * 60 

def get_natural_sort_key(path):
    """Sorts by full directory path first, then naturally by chapter numbers."""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', str(path))]

def load_ledger():
    if os.path.exists(LEDGER_FILE):
        with open(LEDGER_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_ledger(ledger_data):
    with open(LEDGER_FILE, 'w', encoding='utf-8') as f:
        json.dump(ledger_data, f, indent=4)

def process_dataset(input_base, output_base):
    start_time = time.time()
    input_path = Path(input_base)
    output_path = Path(output_base)
    
    # Mirror manifest files
    manifests = list(input_path.rglob("_manifest.json"))
    for manifest in manifests:
        rel_path = manifest.relative_to(input_path)
        out_manifest_path = output_path / rel_path
        out_manifest_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(manifest, out_manifest_path)
        
    pdf_files = list(input_path.rglob("*.pdf"))
    if not pdf_files:
        print("Error: No PDFs found.")
        return

    pdf_files = sorted(pdf_files, key=get_natural_sort_key)
    print(f"Found {len(pdf_files)} PDFs. Processing in serialized order.")

    ledger = load_ledger()

    print("Initializing Marker pipeline...")
    config = {
        "output_format": "json",
        "extract_images": True # Keeps them inside the JSON as base64
    }
    
    config_parser = ConfigParser(config)
    converter = PdfConverter(
        config=config_parser.generate_config_dict(),
        artifact_dict=create_model_dict(),
        processor_list=config_parser.get_processors(),
        renderer=config_parser.get_renderer()
    )

    for pdf_file in pdf_files:
        # --- THE KILL SWITCH ---
        elapsed_time = time.time() - start_time
        if elapsed_time > MAX_RUNTIME_SECONDS:
            print(f"\n[WARNING] Reached 10-hour limit. Shutting down cleanly to save progress.")
            break

        rel_path = pdf_file.relative_to(input_path)
        rel_path_str = str(rel_path)
        
        if ledger.get(rel_path_str) == "Success":
            print(f"Skipping {rel_path.stem}, already verified in ledger.")
            continue
            
        out_dir = output_path / rel_path.parent / rel_path.stem
        out_dir.mkdir(parents=True, exist_ok=True)
        out_json_path = out_dir / f"{rel_path.stem}.json"
        
        print(f"Processing: {rel_path}")
        
        try:
            rendered = converter(str(pdf_file))
            # We don't even capture the physical 'images' dictionary anymore
            marker_json, _, _ = text_from_rendered(rendered)
            
            with open(out_json_path, "w", encoding="utf-8") as f:
                f.write(marker_json)
                            
            print(f"Success: Stored in {out_dir}")
            ledger[rel_path_str] = "Success"
            save_ledger(ledger)
            
        except Exception as e:
            error_msg = str(e)
            print(f"FAILED on {rel_path}: {error_msg}")
            ledger[rel_path_str] = f"Error: {error_msg}" 
            save_ledger(ledger)

if __name__ == "__main__":
    process_dataset(INPUT_DIR, OUTPUT_DIR)

In [ ]:
!python extract_nios.py